In [16]:
import numpy as np
from xgboost import XGBClassifier #for Random Forest
from sklearn.metrics import accuracy_score #evaluation metrics
from tensorflow.keras.models import Sequential #for Feed forward and CNN
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.utils import to_categorical #for softmax need encoding of integers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import mnist #MNIST dataset itself
from sklearn.model_selection import train_test_split

In [10]:
#abstract class that can only be used through subclasses each of which would have train and predict functions
from abc import ABC, abstractmethod

class MnistClassifierInterface(ABC):
    @abstractmethod
    def train(self, X_train, y_train):
        pass

    @abstractmethod
    def predict(self, X_test):
        pass

In [11]:
#XGBoost classifier
class XGBoostMnistClassifier(MnistClassifierInterface):
    def __init__(self):
        self.model = XGBClassifier(
            n_estimators=100,   #number of rounds
            use_label_encoder=False, #old encoding in XGBoost was enabled by default
            eval_metric='mlogloss', #multiclass logarithmic loss, can also use merror
            verbosity=0 #for notebook
        )

    def train(self, X_train, y_train):
        X_train = X_train.reshape(-1, 28*28).astype(np.float32)
        self.model.fit(X_train, y_train)

    def predict(self, X_test):
        X_test = X_test.reshape(-1, 28*28).astype(np.float32)
        return self.model.predict(X_test)

In [12]:
#FNN
class FeedForwardMnistClassifier(MnistClassifierInterface):
    def __init__(self):
        self.model = None

    def train(self, X_train, y_train):
        X_train = X_train.reshape(-1, 28*28).astype('float32') / 255.0 #FNN expects 1D vector, while MNIST has 2D vectors
        y_train = to_categorical(y_train, 10) #encodes integers one-hot

        self.model = Sequential([
            Dense(128, activation='relu', input_shape=(784,)),
            Dropout(0.5),
            Dense(64, activation='relu'),
            Dropout(0.5),
            Dense(10, activation='softmax')
        ])

        self.model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
        self.model.fit(X_train, y_train, batch_size=128, epochs=10, verbose=0)

    def predict(self, X_test):
        X_test = X_test.reshape(-1, 28*28).astype('float32') / 255.0
        predictions = self.model.predict(X_test, verbose=0)
        return np.argmax(predictions, axis=1)

In [13]:
#CNN
class CNNMnistClassifier(MnistClassifierInterface):
    def __init__(self):
        self.model = None

    def train(self, X_train, y_train):
        X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0 #CNN works with 2D vector so no flattening needed
        y_train = to_categorical(y_train, 10) #encodes integers one-hot

        self.model = Sequential([
            Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
            MaxPooling2D(pool_size=(2, 2)),
            Dropout(0.25),

            Conv2D(64, (3, 3), activation='relu'),
            MaxPooling2D(pool_size=(2, 2)),
            Dropout(0.25),

            Flatten(),
            Dense(128, activation='relu'),
            Dropout(0.5),
            Dense(10, activation='softmax')
        ])

        self.model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
        self.model.fit(X_train, y_train, batch_size=128, epochs=10, verbose=0)

    def predict(self, X_test):
        X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
        predictions = self.model.predict(X_test, verbose=0)
        return np.argmax(predictions, axis=1)

In [14]:
#wrapper class
class MnistClassifier:
    def __init__(self, algorithm):
        if algorithm == 'rf':
            self.model = XGBoostMnistClassifier()
        elif algorithm == 'nn':
            self.model = FeedForwardMnistClassifier()
        elif algorithm == 'cnn':
            self.model = CNNMnistClassifier()
        else:
            raise ValueError("Unknown algorithm. Choose from 'rf', 'nn', 'cnn'.")

    def train(self, X_train, y_train):
        self.model.train(X_train, y_train)

    def predict(self, X_test):
        return self.model.predict(X_test)

In [17]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

for option in ['rf', 'nn', 'cnn']:
    print(f"\nTraining and testing model: {option}")
    clf = MnistClassifier(option)
    clf.train(X_train_split, y_train_split)
    val_preds = clf.predict(X_val)
    val_acc = accuracy_score(y_val, val_preds)
    print(f"Validation Accuracy for {option}: {val_acc:.4f}")


Training and testing model: rf
Validation Accuracy for rf: 0.9772

Training and testing model: nn


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Validation Accuracy for nn: 0.9687

Training and testing model: cnn


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Validation Accuracy for cnn: 0.9893
